Implementazione di riferimento del Variational AutoEncoder VAE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_log = nn.Linear(hidden_dim, latent_dim) # log sigma^2

        # Decoder
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_log(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var) # sigma
        eps = torch.randn_like(std) # epsilon ~ N(0,I)
        return mu + eps * std # z = mu + sigma * eps

    def decode(self, z):
        h = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

def vae_loss(x_hat, x, mu, log_var):
    # Termine di ricostruzione (BCE)
    recon = F.binary_cross_entropy(x_hat, x, reduction='sum')

    # Termine KL (formula analitica, eq. 4)
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

    return recon + kl